#### ── Notebook Info ──────────────────────────────────────
#### Repo   : github.com/ShamsherSingh85/AIE_2026
#### Phase  : 1 | Week : 1 | Day : 2
#### Topic  : BERT vs GPT
#### Saved  : File → Save a copy in GitHub
#### ───────────────────────────────────────────────────────

In [ ]:
!pip install transformers torch
!pip install nbstripout

### BERT - predicting masked words

In [ ]:
from transformers import pipeline

# BERT's actual training task - to predict mask words
fill_mask = pipeline('fill-mask', model = 'bert-base-uncased')

sentences = sentences = [
    "The best programming language for AI is [MASK].",
    "To become an AI engineer, you must [MASK] every day.",
    "The capital of France is [MASK]."
]

for sentence in sentences:
  results = fill_mask(sentence)
  print(f"\nSentence: {sentence}")
  print("Top 3 predictions:")
  for r in results[:3]:
    print(f"  '{r['token_str']:<12}' → confidence: {r['score']:.2%}")


### GPT-2 predicting next tokens

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2')

prompt = 'The best way to become an AI Engineer is to'
inputs = tokenizer(prompt, return_tensors = 'pt')

# get the probability distribution
with torch.no_grad():
  outputs = model(**inputs, labels = inputs['input_ids'])
  logits = outputs.logits

# last position = prediction for the next token
next_token_logits = logits[0, -1, :]
probs = torch.softmax(next_token_logits, dim = -1)

# Top 5 predictions
top5 = torch.topk(probs, 5)
print(f"Prompt: '{prompt}'\n")
print("GPT-2 top 5 predictions for next token:")
for prob, idx in zip(top5.values, top5.indices):
  token = tokenizer.decode([idx])
  print(f"  '{token:<12}' → {prob:.2%}")

This is the core loop of every GPT model. **Logits → softmax → sample a token → append to sequence → repeat. That's text generation.**

### Compare BERT vs GPT-2 Embeddings

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

def get_embedding(text, model_name):
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModel.from_pretrained(model_name)

  inputs = tokenizer(text, return_tensors = 'pt', truncation = True)
  with torch.no_grad():
    outputs = model(**inputs)

  embeddings = outputs.last_hidden_state.mean(dim = 1).squeeze()
  return embeddings

def cosine_similarity(a, b):
  return (a @ b / (a.norm() * b.norm())).item()

# Three sentences — 2 similar, 1 different
s1 = "I love machine learning and AI."
s2 = "Artificial intelligence and ML are my passion."
s3 = "The weather today is sunny and warm."

print("── BERT embeddings ──")
e1 = get_embedding(s1, "bert-base-uncased")
e2 = get_embedding(s2, "bert-base-uncased")
e3 = get_embedding(s3, "bert-base-uncased")
print(f"s1 vs s2 (should be HIGH) : {cosine_similarity(e1, e2):.4f}")
print(f"s1 vs s3 (should be LOW)  : {cosine_similarity(e1, e3):.4f}")

print("\n── GPT-2 embeddings ──")
g1 = get_embedding(s1, "gpt2")
g2 = get_embedding(s2, "gpt2")
g3 = get_embedding(s3, "gpt2")
print(f"s1 vs s2 (should be HIGH) : {cosine_similarity(g1, g2):.4f}")
print(f"s1 vs s3 (should be LOW)  : {cosine_similarity(g1, g3):.4f}")

**Key observation:** BERT's similarity scores will be more meaningful than GPT-2's. BERT was designed to understand — GPT-2 was designed to generate. This is why BERT-family models are used for embeddings, not GPT.